# Data Import and Cleaning: U.S. COVID-19 Cases and Deaths

**CPSC 375 Project Check-In** · Team: *Troy Chin, Mohammed Khalaf, Daniel Boynton*

**Project:** Modeling and Forecasting COVID-19 Dynamics in the United States Using Logistic Growth Models

This notebook imports the Our World in Data (OWID) COVID-19 cumulative case and death counts, checks data quality, reshapes the data into tidy format, and builds a cleaned U.S. table.

**Data source:** Mathieu, E., Ritchie, H., Rodés-Guirao, L., Appel, C., Gavrilov, D., Giattino, C., Hasell, J., Macdonald, B., Dattani, S., Beltekian, D., Ortiz-Ospina, E., & Roser, M. (2020). *Coronavirus Pandemic (COVID-19)*. Our World in Data. https://ourworldindata.org/coronavirus

**How to run:** Open in Google Colab (default Python 3.12 runtime) and choose *Runtime → Run all*. No installs or uploads are needed; the data is read directly from the project's GitHub repository.

### Imports

%pip install -q lets-plot

In [36]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

### Loading the Raw Data
- total_cases.csv: cumulative confirmed cases
- total_deaths.csv: cumulative confirmed deaths

In [25]:
URL = "https://raw.githubusercontent.com/troyc03/cpsc-375-project/main/data/"

cases_raw = pd.read_csv(URL + "total_cases.csv", parse_dates=["date"])
deaths_raw = pd.read_csv(URL + "total_deaths.csv", parse_dates=["date"])

cases_raw.head()

,date,World,Afghanistan,Africa,Albania,Algeria,American Samoa,Andorra,Angola,Anguilla,...,Uruguay,Uzbekistan,Vanuatu,Vatican,Venezuela,Vietnam,Wallis and Futuna,Yemen,Zambia,Zimbabwe
0,2020-01-05,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2020-01-06,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2020-01-07,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2020-01-08,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2020-01-09,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Inspect the Raw Data
Checks the date range, missing values, and whether the two files line up.

In [26]:
for name, d in [("cases", cases_raw), ("deaths", deaths_raw)]:
    print(f"{name:>6}: {d.shape[0]} rows x {d.shape[1]} columns, "
          f"{d['date'].min().date()} to {d['date'].max().date()}, "
          f"missing values = {d.isna().sum()}")

print("\nGaps between consecutive dates in days:",
      cases_raw["date"].diff().dropna().dt.days.unique())
print("Same dates in both files:", cases_raw["date"].equals(deaths_raw["date"]))
print("Same columns in both files:", list(cases_raw.columns) == list(deaths_raw.columns))

 cases: 1674 rows x 247 columns, 2020-01-05 to 2024-08-04, missing values = date                 0
World                0
Afghanistan          0
Africa               0
Albania              0
                    ..
Vietnam              0
Wallis and Futuna    0
Yemen                0
Zambia               0
Zimbabwe             0
Length: 247, dtype: int64
deaths: 1674 rows x 247 columns, 2020-01-05 to 2024-08-04, missing values = date                 0
World                0
Afghanistan          0
Africa               0
Albania              0
                    ..
Vietnam              0
Wallis and Futuna    0
Yemen                0
Zambia               0
Zimbabwe             0
Length: 247, dtype: int64

Gaps between consecutive dates in days: [1]
Same dates in both files: True
Same columns in both files: True


Checking which days the counts actually change to see how frequently it's updated.

In [27]:
america_changes = cases_raw.loc[cases_raw["United States"].diff() > 0, "date"]
print("Weekdays on which American cumulative cases change:")
print(america_changes.dt.day_name().value_counts())

# Repeated for every location
changed = cases_raw.set_index("date").diff().gt(0).any(axis=1)
print("\nWeekday when ANY location's cases change:",
      changed[changed].index.day_name().unique().tolist())

Weekdays on which American cumulative cases change:
date
Sunday    173
Name: count, dtype: int64

Weekday when ANY location's cases change: ['Sunday']


**Summary:**
Both cases and deaths datasets cover the same dates at the same frequency (January 2020 to August 2024 daily on Sundays).
The things that need to be fixed to make it tidy are:
1. Case count is spread across lots of columns
2. The location is a variable stored in column headers
3. Not every location value is a country so they have to be separated to not double-count

### Tidying the Data
First, we melt the rows by week instead of by day

In [28]:
cases_wk = cases_raw[cases_raw["date"].dt.day_of_week == 6]   # (Sunday)
deaths_wk = deaths_raw[deaths_raw["date"].dt.day_of_week == 6]

cases_long = cases_wk.melt(
    id_vars="date",
    var_name = "location",
    value_name="total_cases"
)
deaths_long = deaths_wk.melt(
    id_vars="date",
    var_name="location",
    value_name="total_deaths"
)

tidy = (
    cases_long.merge(
        deaths_long,
        on=["date", "location"],
        how="inner",
        validate="one_to_one"
    ).rename(
        columns={"date": "week_ending"}
    ).sort_values(
        ["location", "week_ending"]
    ).reset_index(drop=True)
)
tidy.head()

,week_ending,location,total_cases,total_deaths
0,2020-01-05,Afghanistan,0,0
1,2020-01-12,Afghanistan,0,0
2,2020-01-19,Afghanistan,0,0
3,2020-01-26,Afghanistan,0,0
4,2020-02-02,Afghanistan,0,0


Next, we add a column that flags any regions that overlap in the locations variable (World/European Union...)

In [29]:
AGGREGATES = [
    "World", "Africa", "Asia", "Europe", "North America",
    "South America", "Oceania", "European Union (27)",
    "High-income countries", "Upper-middle-income countries",
    "Lower-middle-income countries", "Low-income countries",
]
tidy["is_aggregate"] = tidy["location"].isin(AGGREGATES)

# Stores counts as integers (measuring whole people)
tidy[["total_cases", "total_deaths"]] = tidy[["total_cases", "total_deaths"]]\
    .astype("int64")

print(tidy.shape)
print("Weeks:", tidy["week_ending"].nunique())
print(
    "Countriesk/territories:",
    tidy.loc[~tidy["is_aggregate"], "location"].nunique()
)
print(
    "Aggregate regions",
    tidy.loc[tidy["is_aggregate"], "location"].nunique()
)
tidy.dtypes

(59040, 5)
Weeks: 240
Countriesk/territories: 234
Aggregate regions 12


week_ending     datetime64[us]
location                   str
total_cases              int64
total_deaths             int64
is_aggregate              bool
dtype: object

Because a cumulative count should never go down, we can also check for decreases meaning a country revised its totals.

In [30]:
for col in ["total_cases", "total_deaths"]:
    drops = tidy.groupby("location")[col].diff() < 0
    print(
        f"{col}: {drops.sum()} downard revisions in",
        tidy.loc[drops, "location"].nunique(), "locations;",
        "United States affected:",
        "United States" in set(tidy.loc[drops, "location"])
    )

total_cases: 20 downard revisions in 20 locations; United States affected: False
total_deaths: 15 downard revisions in 15 locations; United States affected: False


Many countries have small downward revisions, but the US has none. If we do use other countries' data, we'll have to handle them.

### Building the US Table Analysis


In [31]:
us = (tidy[tidy["location"] == "United States"]\
                    .drop(columns=["location", "is_aggregate"])\
                    .reset_index(drop=True))

us = us[us["total_cases"] > 0]\
    .reset_index(drop=True)

us["new_cases"] = us["total_cases"]\
    .diff().fillna(
        us["total_cases"].astype("int64")
    )
us["new_deaths"] = us["total_deaths"]\
    .diff().fillna(
        us["total_deaths"].astype("int64")
    )
us["week"] = np.arange(len(us))

us.head(10)

,week_ending,total_cases,total_deaths,new_cases,new_deaths,week
0,2020-01-26,7,0,7.0,0.0,0
1,2020-02-02,9,0,2.0,0.0,1
2,2020-02-09,20,0,11.0,0.0,2
3,2020-02-16,24,0,4.0,0.0,3
4,2020-02-23,40,0,16.0,0.0,4
5,2020-03-01,73,1,33.0,1.0,5
6,2020-03-08,464,19,391.0,18.0,6
7,2020-03-15,3929,69,3465.0,50.0,7
8,2020-03-22,31415,374,27486.0,305.0,8
9,2020-03-29,126309,2106,94894.0,1732.0,9


### Where is the break in US case reporting?
There was a break where case count stops and deaths keeps going. We'll find the last week with new reported cases

In [34]:
last_case_week = us.loc[us["new_cases"] > 0, "week_ending"].max()
last_deaths_week = us.loc[us["new_deaths"] > 0, "week_ending"].max()
print("Last week with new US cases:", last_case_week.date())
print("Last week with new US deaths:", last_deaths_week.date())
print(
    "Weeks after case reporting stopped:",
    (us["week_ending"] > last_case_week).sum()
)

us["cases_reported"] = us["week_ending"] <= last_case_week

Last week with new US cases: 2023-05-14
Last week with new US deaths: 2024-08-04
Weeks after case reporting stopped: 64


For US case reporting to the WHO, it ended on 05/14/2023. After that week, the cumulative case total is frozen, so the zero "new cases" are missing data and not actually 0. We'll just flag those weeks and exclude them from the analysis.

### Preview Plots
These plots are just to show that the data looks normal.

In [41]:
plot_df = us.assign(
    cumulative_cases = lambda df: df.total_cases / 1000000,
    weekly_new_cases = lambda df: df.new_cases / 1000000,
    weekly_new_deaths = lambda df: df.new_deaths / 1000
).melt(
    id_vars=["week_ending", "cases_reported"],
    value_vars=["cumulative_cases", "weekly_new_cases", "weekly_new_deaths"],
    var_name="measure",
    value_name="value"
)

plot_df["measure"] = plot_df["measure"].map({
    "cumulative_cases": "Cumulate cases (millions)",
    "weekly_new_cases": "New cases per week (millions)",
    "weekly_new_deaths": "New deaths per week (thousands)"
})

# keeps deaths for every week, but cases only during the reported window
plot_df = plot_df[(plot_df["measure"] == "weekly_new_deaths") | plot_df["cases_reported"]]

ggplot(data=plot_df) \
    + geom_line(
        mapping=aes(
            x="week_ending",
            y="value"
        )
    ) \
    + facet_wrap(
        facets="measure",
        nrow=3,
        scales="free_y"
    ) \
    + labs(
        x="Week ending",
        y="Count",
        title="US reported COVID-19 cases and deaths by week"
    )

### Saving the Data

In [42]:
tidy.to_csv("tidy_covid_all_locations.csv", index=False)
us.to_csv("us_covid_clean.csv", index=False)